# Badger_vision Inference Demo

Run inference on images using a trained Badger_vision model.

Supports loading from:
- A YAML model config (uses default weights)
- A saved checkpoint (`checkpoint_best.pt`)

## 1 — Setup

In [ ]:
import subprocess
import sys

for pkg in ["tqdm"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch

print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU     : {gpu_name}  ({gpu_mem:.1f} GB)")
    torch.backends.cudnn.benchmark = True
    DEVICE = torch.device("cuda:0")
else:
    print("Device  : CPU")
    DEVICE = torch.device("cpu")

## 2 — Configure

Point to your model config and the image(s) to run inference on.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE HERE                         ║
# ╚════════════════════════════════════════════════════════════╝

MODEL_CONFIG = "/path/to/model_config.yaml"  # YAML config file
IMAGE_PATH   = "/path/to/image.jpg"           # single image or folder
CHECKPOINT   = None                            # checkpoint .pt path, or None

## 3 — Load Model

In [ ]:
from pathlib import Path

from badger_vision.core.api import Badger_vision

assert Path(MODEL_CONFIG).exists(), f"Config not found: {MODEL_CONFIG}"

model = Badger_vision(MODEL_CONFIG)
print(f"Model loaded from: {MODEL_CONFIG}")

if CHECKPOINT and Path(CHECKPOINT).exists():
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    if "model_state" in ckpt:
        model.model.load_state_dict(ckpt["model_state"])
    elif "state_dict" in ckpt:
        model.model.load_state_dict(ckpt["state_dict"])
    print(f"Checkpoint loaded: {CHECKPOINT}")

## 4 — Run Inference

In [ ]:
import glob

image_path = Path(IMAGE_PATH)
assert image_path.exists(), f"Image not found: {IMAGE_PATH}"

if image_path.is_dir():
    images = sorted(glob.glob(str(image_path / "*.jpg")) +
                    glob.glob(str(image_path / "*.png")))
    print(f"Found {len(images)} images in {image_path}")
else:
    images = [str(image_path)]

for img_path in images[:10]:  # show first 10
    print(f"\n--- {Path(img_path).name} ---")
    preds = model.predict(img_path)
    for key, val in preds.items():
        if hasattr(val, 'shape'):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        else:
            print(f"  {key}: {val}")

## 5 — Benchmark

In [ ]:
results = model.benchmark(warmup=5, runs=50)
print(f"Latency : {results['avg_latency_ms']} ms")
print(f"FPS     : {results['fps']}")
print(f"Params  : {results['params_M']}M")
print(f"FLOPs   : {results['flops_G']} GFLOPs")

## 6 — Export to ONNX

In [ ]:
import os

model.export(format="onnx")
size_mb = os.path.getsize("model.onnx") / (1024 * 1024)
print(f"Exported model.onnx ({size_mb:.1f} MB)")